# Capa TRANSFORMED
Lee `extracted_<etiqueta>` (nunca `raw_`), aplica las reglas de limpieza, mide la calidad antes y después, carga a Parquet, guarda el lineage y valida con DuckDB.
Se trabajan los **3 datasets**: `adilshamim8`, `samithsachidanandan` y `saurabhbadole`.

In [1]:
import sys, os, re, json, sqlite3
import pandas as pd
from datetime import datetime
from IPython.display import display

sys.path.append(os.path.expanduser('~/Documents/IA/4_semestre/ing_de_datos/ing_datos/tools'))
import tools

VERSION_TRANSFORM = "v1.0"
T = tools.transformacion

for p in (tools.RUTA_TRANSF_PROCESSED, tools.RUTA_TRANSF_LOGS, tools.RUTA_TRANSF_REPORTES):
    p.mkdir(parents=True, exist_ok=True)

print("Datasets:", tools.ETIQUETAS)

Datasets: ['adilshamim8', 'samithsachidanandan', 'saurabhbadole']


## 1. Reglas de calidad y de limpieza

Los tres datasets son encuestas de salarios en ciencia de datos y comparten el **mismo esquema de 11 columnas**. Por eso las reglas base son las mismas, aunque se miden y se aplican por separado a cada dataset (`REGLAS[etiqueta]`).

**Completitud.** Las columnas obligatorias son las que permiten analizar un salario: año, nivel de experiencia, tipo de empleo, salario, moneda y salario en USD. Si una de ellas falta, la fila se **elimina**. Las demás columnas se **marcan** o se **imputan** (ver `NULOS_OPCIONALES`).

**Unicidad.** Se buscan duplicados exactos en las 11 columnas de negocio. No se usan los metadatos, porque `_row_hash` y `_batch_id` coinciden en filas iguales.

**Consistencia.** Hay una regla **entre columnas**: si `salary_currency = 'USD'`, entonces `salary` debe ser igual a `salary_in_usd`. También hay reglas de **dominio**: códigos válidos de experiencia, empleo y tamaño, `remote_ratio ∈ {0, 50, 100}`, año entre 2020 y 2025, salarios positivos y países con código ISO-2.

In [2]:
COLS_NEGOCIO = ['work_year', 'experience_level', 'employment_type', 'job_title', 'salary',
                'salary_currency', 'salary_in_usd', 'employee_residence', 'remote_ratio',
                'company_location', 'company_size']

COLS_TEXTO    = ['experience_level', 'employment_type', 'job_title', 'salary_currency',
                 'employee_residence', 'company_location', 'company_size']
COLS_CODIGOS  = ['experience_level', 'employment_type', 'salary_currency',
                 'employee_residence', 'company_location', 'company_size']   # se pasan a MAYÚSCULAS

OBLIGATORIAS  = ['work_year', 'experience_level', 'employment_type', 'salary', 'salary_currency', 'salary_in_usd']

# columna -> (decisión, valor de reemplazo, justificación)
NULOS_OPCIONALES = {
    'job_title':          ('marcar',  'Sin especificar', 'El puesto describe la fila pero no invalida el salario; se marca para no perder el registro.'),
    'employee_residence': ('marcar',  'ND',              'País desconocido; se marca con ND para no inventar una ubicación.'),
    'company_location':   ('marcar',  'ND',              'País desconocido; se marca con ND para no inventar una ubicación.'),
    'company_size':       ('marcar',  'ND',              'Tamaño desconocido; se marca con ND.'),
    'remote_ratio':       ('imputar', 0,                 'Se imputa con la moda (0 = presencial), que es la mayoría de los registros.'),
}

REGLAS_CONSISTENCIA = {
    'usd_salary_distinto_de_salary_in_usd': lambda d: (d['salary_currency'] == 'USD') & (d['salary'] != d['salary_in_usd']),
    'experience_level_invalido':            lambda d: ~d['experience_level'].isin(['EN', 'MI', 'SE', 'EX']),
    'employment_type_invalido':             lambda d: ~d['employment_type'].isin(['FT', 'PT', 'CT', 'FL']),
    'company_size_invalido':                lambda d: ~d['company_size'].isin(['S', 'M', 'L']),
    'remote_ratio_fuera_de_0_50_100':       lambda d: ~d['remote_ratio'].isin([0, 50, 100]),
    'work_year_fuera_de_2020_2025':         lambda d: ~d['work_year'].between(2020, 2025),
    'salario_no_positivo':                  lambda d: (d['salary'] <= 0) | (d['salary_in_usd'] <= 0),
    'pais_no_iso2':                         lambda d: ~(d['employee_residence'].astype(str).str.fullmatch(r'[A-Z]{2}')
                                                        & d['company_location'].astype(str).str.fullmatch(r'[A-Z]{2}')),
}

# Mismo esquema => mismas reglas base, pero cada dataset tiene su entrada (aquí se ajustarían si difieren)
REGLAS = {et: {'obligatorias': OBLIGATORIAS, 'nulos': NULOS_OPCIONALES, 'consistencia': REGLAS_CONSISTENCIA}
          for et in tools.ETIQUETAS}

## 2. Función de transformación
Cada paso se registra en la bitácora con las filas antes, después, eliminadas y afectadas, junto con su decisión y justificación.

In [3]:
def transformar(etiqueta):
    cfg     = tools.DATASETS[etiqueta]
    reglas  = REGLAS[etiqueta]
    db_path = cfg['db']
    tabla_extracted = f'extracted_{etiqueta}'
    destino  = f'transformed_{etiqueta}'
    parquet  = tools.RUTA_TRANSF_PROCESSED / f'{destino}.parquet'
    lineage  = tools.RUTA_TRANSF_LOGS / f'lineage_{destino}.json'
    bit_csv  = tools.RUTA_TRANSF_REPORTES / f'bitacora_{destino}.csv'
    cal_csv  = tools.RUTA_TRANSF_REPORTES / f'calidad_{destino}.csv'
    log = lambda **kw: tools.logs.escribir_log(tools.LOG_TRANSF, etiqueta=etiqueta, version=VERSION_TRANSFORM, **kw)

    try:
        # ── 0. Leer EXTRACTED en solo lectura ─────────────────────────────────
        conn = sqlite3.connect(f'file:{db_path}?mode=ro', uri=True)
        df_ext = pd.read_sql_query(f'SELECT * FROM {tabla_extracted}', conn)
        conn.close()
        log(etapa='lectura_extracted', origen=tabla_extracted, filas_entrada=len(df_ext), estado='exitoso')

        # ── Calidad ANTES ─────────────────────────────────────────────────────
        calidad_antes = T.medir_calidad(df_ext, COLS_NEGOCIO, reglas['obligatorias'], reglas['consistencia'])

        df  = df_ext.copy()
        bit = []

        # ── 1. Nombres de columnas: espacios -> '_' ───────────────────────────
        nuevos = {c: re.sub(r'\s+', '_', c.strip()) for c in df.columns}
        cambiadas = [c for c, n in nuevos.items() if c != n]
        df = df.rename(columns=nuevos)
        T.registrar(bit, 'renombrar_columnas', len(df), len(df), 0, 'renombrar',
                    f'Se reemplazan espacios por "_" en los nombres; {len(cambiadas)} columnas tenían espacios.', cambiadas)

        # ── 2. Limpieza de texto ──────────────────────────────────────────────
        n = len(df)
        df, afectadas = T.limpiar_texto(df, COLS_TEXTO, mayusculas=COLS_CODIGOS)
        T.registrar(bit, 'limpieza_texto', n, len(df), afectadas, 'normalizar',
                    'strip de espacios al inicio y al final, se colapsan espacios dobles y los códigos se pasan a MAYÚSCULAS para que las categorías coincidan.',
                    COLS_TEXTO)

        # ── 3a. Nulos en obligatorias: ELIMINAR ───────────────────────────────
        n = len(df)
        mask = df[reglas['obligatorias']].isna().any(axis=1)
        df = df[~mask]
        T.registrar(bit, 'nulos_obligatorias', n, len(df), int(mask.sum()), 'eliminar',
                    'Sin año, nivel, tipo de empleo o salario la fila no sirve para el análisis salarial y no se puede imputar sin sesgar.',
                    reglas['obligatorias'])

        # ── 3b. Nulos en opcionales: MARCAR / IMPUTAR ─────────────────────────
        for col, (decision, valor, just) in reglas['nulos'].items():
            n = len(df)
            nulos = int(df[col].isna().sum())
            df[col] = df[col].fillna(valor)
            T.registrar(bit, f'nulos_{col}', n, len(df), nulos, decision, just, [col])

        # ── 4. Duplicados exactos: ELIMINAR ───────────────────────────────────
        n = len(df)
        df = df.drop_duplicates(subset=COLS_NEGOCIO, keep='first')
        T.registrar(bit, 'duplicados_exactos', n, len(df), n - len(df), 'eliminar',
                    'Filas idénticas en las 11 columnas de negocio inflan los conteos y sesgan los promedios; se conserva la primera aparición.',
                    COLS_NEGOCIO)

        # ── 5. Reglas de consistencia/dominio: ELIMINAR violaciones ───────────
        n = len(df)
        mask = pd.Series(False, index=df.index)
        for regla in reglas['consistencia'].values():
            mask |= regla(df).fillna(True)
        df = df[~mask]
        T.registrar(bit, 'reglas_consistencia', n, len(df), int(mask.sum()), 'eliminar',
                    'Una fila que rompe una regla de negocio (moneda USD distinta de salary_in_usd, códigos fuera de catálogo, año o remote_ratio inválidos) no es confiable.',
                    list(reglas['consistencia'].keys()))

        # ── 6. Conversión de tipos ────────────────────────────────────────────
        n = len(df)
        df = df.astype({'work_year': 'int16', 'salary': 'int64', 'salary_in_usd': 'int64',
                        'remote_ratio': 'int8', '_batch_id': 'int32'})
        for c in COLS_TEXTO + ['_source_table', '_row_hash']:
            df[c] = df[c].astype('string')
        df['_extracted_at'] = pd.to_datetime(df['_extracted_at'])
        T.registrar(bit, 'conversion_tipos', n, len(df), len(df), 'convertir',
                    'Enteros: work_year int16, remote_ratio int8, salarios int64. Texto: string. Fecha: _extracted_at a datetime.',
                    ['work_year', 'remote_ratio', 'salary', 'salary_in_usd', '_extracted_at'] + COLS_TEXTO)

        # ── 7. Columnas derivadas: booleanos y decimales ──────────────────────
        n = len(df)
        df['es_remoto_total']   = df['remote_ratio'].eq(100)
        df['misma_ubicacion']   = df['employee_residence'].eq(df['company_location']).astype(bool)
        df['pago_en_usd']       = df['salary_currency'].eq('USD').astype(bool)
        df['tasa_cambio_a_usd'] = (df['salary_in_usd'] / df['salary']).round(6).astype('float64')
        T.registrar(bit, 'columnas_derivadas', n, len(df), len(df), 'derivar',
                    'Booleanos para filtrar (remoto total, empleado en el país de la empresa, pago en USD) y un decimal con la tasa implícita salary_in_usd / salary.',
                    ['es_remoto_total', 'misma_ubicacion', 'pago_en_usd', 'tasa_cambio_a_usd'])

        # ── 8. Atípicos de salario: MARCAR (no eliminar) ──────────────────────
        q1, q3 = df['salary_in_usd'].quantile([0.25, 0.75])
        lim_inf, lim_sup = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
        df['_outlier_salario'] = ~df['salary_in_usd'].between(lim_inf, lim_sup)
        T.registrar(bit, 'marcar_atipicos_salario', len(df), len(df), int(df['_outlier_salario'].sum()), 'marcar',
                    f'Regla IQR sobre salary_in_usd (fuera de [{lim_inf:,.0f}, {lim_sup:,.0f}]). Se marcan y no se eliminan porque pueden ser salarios reales de puestos ejecutivos.',
                    ['salary_in_usd'])

        # ── 9. Metadatos de transformación (se conservan los de extracción) ───
        df['_transformed_at']    = pd.Timestamp.now().floor('s')
        df['_transform_version'] = VERSION_TRANSFORM
        T.registrar(bit, 'metadatos_transformacion', len(df), len(df), len(df), 'agregar',
                    'Se agregan _transformed_at y _transform_version; se conservan _batch_id, _extracted_at, _source_table y _row_hash.',
                    ['_transformed_at', '_transform_version'])

        df_final = df.reset_index(drop=True)
        bitacora = pd.DataFrame(bit)

        # ── Calidad DESPUÉS + cuadre de la bitácora ───────────────────────────
        calidad_despues = T.medir_calidad(df_final, COLS_NEGOCIO, reglas['obligatorias'], reglas['consistencia'])
        calidad = T.tabla_calidad(calidad_antes, calidad_despues)
        eliminadas_bitacora = int(bitacora['filas_eliminadas'].sum())
        eliminadas_reales   = len(df_ext) - len(df_final)
        cuadra = eliminadas_bitacora == eliminadas_reales
        log(etapa='transformacion', filas_entrada=len(df_ext), filas_salida=len(df_final),
            filas_eliminadas_bitacora=eliminadas_bitacora, filas_eliminadas_reales=eliminadas_reales,
            cuadre_bitacora=cuadra, estado='exitoso' if cuadra else 'error')

        # ── Evidencias en Reportes/ ───────────────────────────────────────────
        bitacora.to_csv(bit_csv, index=False, encoding='utf-8')
        calidad.to_csv(cal_csv, index=False, encoding='utf-8')

        # ── Carga a Parquet (sobrescribe) + lectura de vuelta ─────────────────
        df_final.to_parquet(parquet, index=False)
        log(etapa='carga_parquet', origen=tabla_extracted, destino=str(parquet),
            filas_entrada=len(df_ext), filas_salida=len(df_final), estado='exitoso')
        df_leido = pd.read_parquet(parquet)
        tipos_ok = df_leido.dtypes.astype(str).to_dict() == df_final.dtypes.astype(str).to_dict()
        log(etapa='lectura_validacion', filas_entrada=len(df_final), filas_salida=len(df_leido),
            tipos_conservados=tipos_ok, estado='exitoso' if tipos_ok and len(df_leido) == len(df_final) else 'error')

        # ── Lineage ───────────────────────────────────────────────────────────
        tools.logs.crear_cdc(
            lineage, etiqueta, reemplazar=True,
            recorrido=[f'Kaggle ({etiqueta}) CSV', f"RAW: {cfg['raw']} ({db_path.name})",
                       f'EXTRACTED: {tabla_extracted} ({db_path.name})',
                       f'TRANSFORMED: Transformacion_de_Datos/Processed/{destino}.parquet'],
            origen=tabla_extracted, destino=str(parquet.relative_to(tools.BASE_DIR)),
            filas_origen=len(df_ext), filas_destino=len(df_final),
            version_reglas=VERSION_TRANSFORM,
            columnas_obligatorias=reglas['obligatorias'],
            reglas_consistencia=list(reglas['consistencia'].keys()),
            bitacora=bitacora.drop(columns='timestamp').to_dict(orient='records'),
            tipos_columnas=df_final.dtypes.astype(str).to_dict(),
        )

        # ── Validación con DuckDB sobre el archivo cargado ────────────────────
        esquema, conteo = T.validar_duckdb(parquet, reglas['obligatorias'])
        nulos_oblig = int(conteo.drop(columns='filas').sum(axis=1).iloc[0])
        log(etapa='validacion_duckdb', destino=str(parquet), filas_salida=int(conteo['filas'].iloc[0]),
            nulos_obligatorias=nulos_oblig, columnas=len(esquema),
            estado='exitoso' if nulos_oblig == 0 and int(conteo['filas'].iloc[0]) == len(df_final) else 'error')

        print(f'[{etiqueta}] {len(df_ext):,} -> {len(df_final):,} filas | eliminadas bitácora={eliminadas_bitacora:,} '
              f'reales={eliminadas_reales:,} cuadra={cuadra} | tipos conservados={tipos_ok}')
        return dict(df_final=df_final, bitacora=bitacora, calidad=calidad, esquema=esquema, conteo=conteo,
                    cuadra=cuadra, tipos_ok=tipos_ok, filas_ext=len(df_ext))

    except Exception as e:
        log(etapa='error', estado='error', error=str(e))
        print(f'[{etiqueta}] ERROR: {e}')
        raise

## 3. Ejecución para los 3 datasets

In [4]:
resultados = {et: transformar(et) for et in tools.ETIQUETAS}

[adilshamim8] 151,445 -> 71,913 filas | eliminadas bitácora=79,532 reales=79,532 cuadra=True | tipos conservados=True


[samithsachidanandan] 88,584 -> 44,244 filas | eliminadas bitácora=44,340 reales=44,340 cuadra=True | tipos conservados=True


[saurabhbadole] 93,597 -> 46,637 filas | eliminadas bitácora=46,960 reales=46,960 cuadra=True | tipos conservados=True


## Resultados: `adilshamim8`

In [5]:
r = resultados['adilshamim8']
print('Bitácora de transformaciones')
display(r['bitacora'].drop(columns='timestamp'))

Bitácora de transformaciones


,paso,transformacion,columnas,filas_antes,filas_despues,filas_eliminadas,filas_afectadas,decision,justificacion
0,1,renombrar_columnas,,151445,151445,0,0,renombrar,"Se reemplazan espacios por ""_"" en los nombres;..."
1,2,limpieza_texto,"experience_level, employment_type, job_title, ...",151445,151445,0,0,normalizar,"strip de espacios al inicio y al final, se col..."
2,3,nulos_obligatorias,"work_year, experience_level, employment_type, ...",151445,151445,0,0,eliminar,"Sin año, nivel, tipo de empleo o salario la fi..."
3,4,nulos_job_title,job_title,151445,151445,0,0,marcar,El puesto describe la fila pero no invalida el...
4,5,nulos_employee_residence,employee_residence,151445,151445,0,0,marcar,País desconocido; se marca con ND para no inve...
5,6,nulos_company_location,company_location,151445,151445,0,0,marcar,País desconocido; se marca con ND para no inve...
6,7,nulos_company_size,company_size,151445,151445,0,0,marcar,Tamaño desconocido; se marca con ND.
7,8,nulos_remote_ratio,remote_ratio,151445,151445,0,0,imputar,"Se imputa con la moda (0 = presencial), que es..."
8,9,duplicados_exactos,"work_year, experience_level, employment_type, ...",151445,71913,79532,79532,eliminar,Filas idénticas en las 11 columnas de negocio ...
9,10,reglas_consistencia,"usd_salary_distinto_de_salary_in_usd, experien...",71913,71913,0,0,eliminar,Una fila que rompe una regla de negocio (moned...


In [6]:
print('Calidad antes vs. después')
display(r['calidad'])
print(f"Cuadre bitácora: eliminadas según bitácora = {r['bitacora']['filas_eliminadas'].sum():,} | "
      f"perdidas reales = {r['filas_ext'] - len(r['df_final']):,} | cuadra = {r['cuadra']}")

Calidad antes vs. después


,dimension,metrica,antes,despues,diferencia
0,General,filas,151445,71913,-79532
1,Completitud,nulos_work_year,0,0,0
2,Completitud,nulos_experience_level,0,0,0
3,Completitud,nulos_employment_type,0,0,0
4,Completitud,nulos_job_title,0,0,0
5,Completitud,nulos_salary,0,0,0
6,Completitud,nulos_salary_currency,0,0,0
7,Completitud,nulos_salary_in_usd,0,0,0
8,Completitud,nulos_employee_residence,0,0,0
9,Completitud,nulos_remote_ratio,0,0,0


Cuadre bitácora: eliminadas según bitácora = 79,532 | perdidas reales = 79,532 | cuadra = True


In [7]:
print('DuckDB - DESCRIBE del Parquet')
display(r['esquema'][['column_name', 'column_type', 'null']])
print('DuckDB - conteo y nulos en columnas obligatorias')
display(r['conteo'])

DuckDB - DESCRIBE del Parquet


,column_name,column_type,null
0,work_year,SMALLINT,YES
1,experience_level,VARCHAR,YES
2,employment_type,VARCHAR,YES
3,job_title,VARCHAR,YES
4,salary,BIGINT,YES
5,salary_currency,VARCHAR,YES
6,salary_in_usd,BIGINT,YES
7,employee_residence,VARCHAR,YES
8,remote_ratio,TINYINT,YES
9,company_location,VARCHAR,YES


DuckDB - conteo y nulos en columnas obligatorias


,filas,nulos_work_year,nulos_experience_level,nulos_employment_type,nulos_salary,nulos_salary_currency,nulos_salary_in_usd
0,71913,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
display(r['df_final'].head())

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,...,_extracted_at,_source_table,_row_hash,es_remoto_total,misma_ubicacion,pago_en_usd,tasa_cambio_a_usd,_outlier_salario,_transformed_at,_transform_version
0,2025,EX,FT,Head of Data,348516,USD,348516,US,0,US,...,2026-11-07 22:43:08,data,95b566db6a71c35f,False,True,True,1.0,True,2026-11-07 22:44:35,v1.0
1,2025,EX,FT,Head of Data,232344,USD,232344,US,0,US,...,2026-11-07 22:43:08,data,d3a6b1cd671b1e99,False,True,True,1.0,False,2026-11-07 22:44:35,v1.0
2,2025,SE,FT,Data Scientist,145400,USD,145400,US,0,US,...,2026-11-07 22:43:08,data,7150c9650f025496,False,True,True,1.0,False,2026-11-07 22:44:35,v1.0
3,2025,SE,FT,Data Scientist,81600,USD,81600,US,0,US,...,2026-11-07 22:43:08,data,4ff664fae1d58f98,False,True,True,1.0,False,2026-11-07 22:44:35,v1.0
4,2025,MI,FT,Engineer,160000,USD,160000,US,100,US,...,2026-11-07 22:43:08,data,ecf9cbea6e0b1327,True,True,True,1.0,False,2026-11-07 22:44:35,v1.0


## Resultados: `samithsachidanandan`

In [9]:
r = resultados['samithsachidanandan']
print('Bitácora de transformaciones')
display(r['bitacora'].drop(columns='timestamp'))

Bitácora de transformaciones


,paso,transformacion,columnas,filas_antes,filas_despues,filas_eliminadas,filas_afectadas,decision,justificacion
0,1,renombrar_columnas,,88584,88584,0,0,renombrar,"Se reemplazan espacios por ""_"" en los nombres;..."
1,2,limpieza_texto,"experience_level, employment_type, job_title, ...",88584,88584,0,0,normalizar,"strip de espacios al inicio y al final, se col..."
2,3,nulos_obligatorias,"work_year, experience_level, employment_type, ...",88584,88584,0,0,eliminar,"Sin año, nivel, tipo de empleo o salario la fi..."
3,4,nulos_job_title,job_title,88584,88584,0,0,marcar,El puesto describe la fila pero no invalida el...
4,5,nulos_employee_residence,employee_residence,88584,88584,0,0,marcar,País desconocido; se marca con ND para no inve...
5,6,nulos_company_location,company_location,88584,88584,0,0,marcar,País desconocido; se marca con ND para no inve...
6,7,nulos_company_size,company_size,88584,88584,0,0,marcar,Tamaño desconocido; se marca con ND.
7,8,nulos_remote_ratio,remote_ratio,88584,88584,0,0,imputar,"Se imputa con la moda (0 = presencial), que es..."
8,9,duplicados_exactos,"work_year, experience_level, employment_type, ...",88584,44244,44340,44340,eliminar,Filas idénticas en las 11 columnas de negocio ...
9,10,reglas_consistencia,"usd_salary_distinto_de_salary_in_usd, experien...",44244,44244,0,0,eliminar,Una fila que rompe una regla de negocio (moned...


In [10]:
print('Calidad antes vs. después')
display(r['calidad'])
print(f"Cuadre bitácora: eliminadas según bitácora = {r['bitacora']['filas_eliminadas'].sum():,} | "
      f"perdidas reales = {r['filas_ext'] - len(r['df_final']):,} | cuadra = {r['cuadra']}")

Calidad antes vs. después


,dimension,metrica,antes,despues,diferencia
0,General,filas,88584,44244,-44340
1,Completitud,nulos_work_year,0,0,0
2,Completitud,nulos_experience_level,0,0,0
3,Completitud,nulos_employment_type,0,0,0
4,Completitud,nulos_job_title,0,0,0
5,Completitud,nulos_salary,0,0,0
6,Completitud,nulos_salary_currency,0,0,0
7,Completitud,nulos_salary_in_usd,0,0,0
8,Completitud,nulos_employee_residence,0,0,0
9,Completitud,nulos_remote_ratio,0,0,0


Cuadre bitácora: eliminadas según bitácora = 44,340 | perdidas reales = 44,340 | cuadra = True


In [11]:
print('DuckDB - DESCRIBE del Parquet')
display(r['esquema'][['column_name', 'column_type', 'null']])
print('DuckDB - conteo y nulos en columnas obligatorias')
display(r['conteo'])

DuckDB - DESCRIBE del Parquet


,column_name,column_type,null
0,work_year,SMALLINT,YES
1,experience_level,VARCHAR,YES
2,employment_type,VARCHAR,YES
3,job_title,VARCHAR,YES
4,salary,BIGINT,YES
5,salary_currency,VARCHAR,YES
6,salary_in_usd,BIGINT,YES
7,employee_residence,VARCHAR,YES
8,remote_ratio,TINYINT,YES
9,company_location,VARCHAR,YES


DuckDB - conteo y nulos en columnas obligatorias


,filas,nulos_work_year,nulos_experience_level,nulos_employment_type,nulos_salary,nulos_salary_currency,nulos_salary_in_usd
0,44244,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
display(r['df_final'].head())

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,...,_extracted_at,_source_table,_row_hash,es_remoto_total,misma_ubicacion,pago_en_usd,tasa_cambio_a_usd,_outlier_salario,_transformed_at,_transform_version
0,2025,MI,FT,Customer Success Manager,57000,EUR,60000,NL,50,NL,...,2026-11-07 22:43:09,data,261674dcbcdcf1a4,False,True,False,1.052632,False,2026-11-07 22:44:37,v1.0
1,2025,SE,FT,Engineer,165000,USD,165000,US,0,US,...,2026-11-07 22:43:09,data,a3b0ad9afbcbbed8,False,True,True,1.000000,False,2026-11-07 22:44:37,v1.0
2,2025,SE,FT,Engineer,109000,USD,109000,US,0,US,...,2026-11-07 22:43:09,data,55455f9cefa2ba22,False,True,True,1.000000,False,2026-11-07 22:44:37,v1.0
3,2025,SE,FT,Applied Scientist,294000,USD,294000,US,0,US,...,2026-11-07 22:43:09,data,be32c065457034c1,False,True,True,1.000000,False,2026-11-07 22:44:37,v1.0
4,2025,SE,FT,Applied Scientist,137600,USD,137600,US,0,US,...,2026-11-07 22:43:09,data,c7d733c7bb1d3e25,False,True,True,1.000000,False,2026-11-07 22:44:37,v1.0


## Resultados: `saurabhbadole`

In [13]:
r = resultados['saurabhbadole']
print('Bitácora de transformaciones')
display(r['bitacora'].drop(columns='timestamp'))

Bitácora de transformaciones


,paso,transformacion,columnas,filas_antes,filas_despues,filas_eliminadas,filas_afectadas,decision,justificacion
0,1,renombrar_columnas,,93597,93597,0,0,renombrar,"Se reemplazan espacios por ""_"" en los nombres;..."
1,2,limpieza_texto,"experience_level, employment_type, job_title, ...",93597,93597,0,0,normalizar,"strip de espacios al inicio y al final, se col..."
2,3,nulos_obligatorias,"work_year, experience_level, employment_type, ...",93597,93597,0,0,eliminar,"Sin año, nivel, tipo de empleo o salario la fi..."
3,4,nulos_job_title,job_title,93597,93597,0,0,marcar,El puesto describe la fila pero no invalida el...
4,5,nulos_employee_residence,employee_residence,93597,93597,0,0,marcar,País desconocido; se marca con ND para no inve...
5,6,nulos_company_location,company_location,93597,93597,0,0,marcar,País desconocido; se marca con ND para no inve...
6,7,nulos_company_size,company_size,93597,93597,0,0,marcar,Tamaño desconocido; se marca con ND.
7,8,nulos_remote_ratio,remote_ratio,93597,93597,0,0,imputar,"Se imputa con la moda (0 = presencial), que es..."
8,9,duplicados_exactos,"work_year, experience_level, employment_type, ...",93597,46637,46960,46960,eliminar,Filas idénticas en las 11 columnas de negocio ...
9,10,reglas_consistencia,"usd_salary_distinto_de_salary_in_usd, experien...",46637,46637,0,0,eliminar,Una fila que rompe una regla de negocio (moned...


In [14]:
print('Calidad antes vs. después')
display(r['calidad'])
print(f"Cuadre bitácora: eliminadas según bitácora = {r['bitacora']['filas_eliminadas'].sum():,} | "
      f"perdidas reales = {r['filas_ext'] - len(r['df_final']):,} | cuadra = {r['cuadra']}")

Calidad antes vs. después


,dimension,metrica,antes,despues,diferencia
0,General,filas,93597,46637,-46960
1,Completitud,nulos_work_year,0,0,0
2,Completitud,nulos_experience_level,0,0,0
3,Completitud,nulos_employment_type,0,0,0
4,Completitud,nulos_job_title,0,0,0
5,Completitud,nulos_salary,0,0,0
6,Completitud,nulos_salary_currency,0,0,0
7,Completitud,nulos_salary_in_usd,0,0,0
8,Completitud,nulos_employee_residence,0,0,0
9,Completitud,nulos_remote_ratio,0,0,0


Cuadre bitácora: eliminadas según bitácora = 46,960 | perdidas reales = 46,960 | cuadra = True


In [15]:
print('DuckDB - DESCRIBE del Parquet')
display(r['esquema'][['column_name', 'column_type', 'null']])
print('DuckDB - conteo y nulos en columnas obligatorias')
display(r['conteo'])

DuckDB - DESCRIBE del Parquet


,column_name,column_type,null
0,work_year,SMALLINT,YES
1,experience_level,VARCHAR,YES
2,employment_type,VARCHAR,YES
3,job_title,VARCHAR,YES
4,salary,BIGINT,YES
5,salary_currency,VARCHAR,YES
6,salary_in_usd,BIGINT,YES
7,employee_residence,VARCHAR,YES
8,remote_ratio,TINYINT,YES
9,company_location,VARCHAR,YES


DuckDB - conteo y nulos en columnas obligatorias


,filas,nulos_work_year,nulos_experience_level,nulos_employment_type,nulos_salary,nulos_salary_currency,nulos_salary_in_usd
0,46637,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
display(r['df_final'].head())

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,...,_extracted_at,_source_table,_row_hash,es_remoto_total,misma_ubicacion,pago_en_usd,tasa_cambio_a_usd,_outlier_salario,_transformed_at,_transform_version
0,2025,MI,FT,Research Scientist,208000,USD,208000,US,0,US,...,2026-11-07 22:43:09,data,d9257db080df5019,False,True,True,1.0,False,2026-11-07 22:44:38,v1.0
1,2025,MI,FT,Research Scientist,147000,USD,147000,US,0,US,...,2026-11-07 22:43:09,data,6de8ed3c6ff836fc,False,True,True,1.0,False,2026-11-07 22:44:38,v1.0
2,2025,SE,FT,Research Scientist,173000,USD,173000,US,0,US,...,2026-11-07 22:43:09,data,a0ae077a49dbd4af,False,True,True,1.0,False,2026-11-07 22:44:38,v1.0
3,2025,SE,FT,Research Scientist,117000,USD,117000,US,0,US,...,2026-11-07 22:43:09,data,ed3266945508012c,False,True,True,1.0,False,2026-11-07 22:44:38,v1.0
4,2025,MI,FT,AI Engineer,100000,USD,100000,US,100,US,...,2026-11-07 22:43:09,data,847048b0d85682d9,True,True,True,1.0,False,2026-11-07 22:44:38,v1.0


## 4. Log de transformación y archivos generados

In [17]:
log = pd.read_json(tools.LOG_TRANSF, lines=True)
display(log)

for et in tools.ETIQUETAS:
    archivos = [tools.RUTA_TRANSF_PROCESSED / f'transformed_{et}.parquet',
                tools.RUTA_TRANSF_LOGS / f'lineage_transformed_{et}.json',
                tools.RUTA_TRANSF_REPORTES / f'bitacora_transformed_{et}.csv',
                tools.RUTA_TRANSF_REPORTES / f'calidad_transformed_{et}.csv']
    print(et, {a.name: a.exists() for a in archivos})

,timestamp,etiqueta,version,etapa,origen,filas_entrada,estado,filas_salida,filas_eliminadas_bitacora,filas_eliminadas_reales,cuadre_bitacora,destino,tipos_conservados,nulos_obligatorias,columnas
0,2026-11-07 22:44:34,adilshamim8,v1.0,lectura_extracted,extracted_adilshamim8,151445.0,exitoso,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-11-07 22:44:35,adilshamim8,v1.0,transformacion,NaN,151445.0,exitoso,71913.0,79532.0,79532.0,1.0,NaN,NaN,NaN,NaN
2,2026-11-07 22:44:36,adilshamim8,v1.0,carga_parquet,extracted_adilshamim8,151445.0,exitoso,71913.0,NaN,NaN,NaN,/Users/memoblack1/Documents/IA/4_semestre/ing_...,NaN,NaN,NaN
3,2026-11-07 22:44:37,adilshamim8,v1.0,lectura_validacion,NaN,71913.0,exitoso,71913.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN
4,2026-11-07 22:44:37,adilshamim8,v1.0,validacion_duckdb,NaN,NaN,exitoso,71913.0,NaN,NaN,NaN,/Users/memoblack1/Documents/IA/4_semestre/ing_...,NaN,0.0,22.0
5,2026-11-07 22:44:37,samithsachidanandan,v1.0,lectura_extracted,extracted_samithsachidanandan,88584.0,exitoso,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2026-11-07 22:44:38,samithsachidanandan,v1.0,transformacion,NaN,88584.0,exitoso,44244.0,44340.0,44340.0,1.0,NaN,NaN,NaN,NaN
7,2026-11-07 22:44:38,samithsachidanandan,v1.0,carga_parquet,extracted_samithsachidanandan,88584.0,exitoso,44244.0,NaN,NaN,NaN,/Users/memoblack1/Documents/IA/4_semestre/ing_...,NaN,NaN,NaN
8,2026-11-07 22:44:38,samithsachidanandan,v1.0,lectura_validacion,NaN,44244.0,exitoso,44244.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN
9,2026-11-07 22:44:38,samithsachidanandan,v1.0,validacion_duckdb,NaN,NaN,exitoso,44244.0,NaN,NaN,NaN,/Users/memoblack1/Documents/IA/4_semestre/ing_...,NaN,0.0,22.0


adilshamim8 {'transformed_adilshamim8.parquet': True, 'lineage_transformed_adilshamim8.json': True, 'bitacora_transformed_adilshamim8.csv': True, 'calidad_transformed_adilshamim8.csv': True}
samithsachidanandan {'transformed_samithsachidanandan.parquet': True, 'lineage_transformed_samithsachidanandan.json': True, 'bitacora_transformed_samithsachidanandan.csv': True, 'calidad_transformed_samithsachidanandan.csv': True}
saurabhbadole {'transformed_saurabhbadole.parquet': True, 'lineage_transformed_saurabhbadole.json': True, 'bitacora_transformed_saurabhbadole.csv': True, 'calidad_transformed_saurabhbadole.csv': True}
